In [0]:
%python
from pathlib import Path
from html.parser import HTMLParser
from urllib.parse import urljoin, urlparse

import urllib.request
import urllib.error
import hashlib
import json
import os
import uuid
CATALOG = "workspace"
SCHEMA = "bls_productivity"
VOLUME = "raw_files"

VOLUME_ROOT = Path(
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
)

BLS_RAW_DIR = (
    VOLUME_ROOT / "current" / "bls"
)

POPULATION_RAW_DIR = (
    VOLUME_ROOT / "current" / "population"
)

BLS_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

POPULATION_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Volume root:", VOLUME_ROOT)
print("BLS folder:", BLS_RAW_DIR)
print("Population folder:", POPULATION_RAW_DIR)

Volume root: /Volumes/workspace/bls_productivity/raw_files
BLS folder: /Volumes/workspace/bls_productivity/raw_files/current/bls
Population folder: /Volumes/workspace/bls_productivity/raw_files/current/population


In [0]:
%python
BLS_URL = (
    "https://download.bls.gov/pub/time.series/pr/"
)

POPULATION_URL = (
    "https://honolulu-api.datausa.io/"
    "tesseract/data.jsonrecords?"
    "cube=acs_yg_total_population_1"
    "&drilldowns=Year%2CNation"
    "&locale=en"
    "&measures=Population"
)

#CONTACT_EMAIL = "akankshabhambhani1@gmail.com"
CONTACT_EMAIL = os.getenv(
    "BLS_CONTACT_EMAIL",
    "data-engineering-contact@example.com"
)

USER_AGENT = (
    "Databricks-BLS-Assessment/1.0 "
    f"({CONTACT_EMAIL})"
)

print(BLS_URL)
print(POPULATION_URL)

https://download.bls.gov/pub/time.series/pr/
https://honolulu-api.datausa.io/tesseract/data.jsonrecords?cube=acs_yg_total_population_1&drilldowns=Year%2CNation&locale=en&measures=Population


In [0]:
%python
def download_url(url):

    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "*/*"
        }
    )

    with urllib.request.urlopen(
        request,
        timeout=120
    ) as response:

        content = response.read()

        return content

In [0]:
%python
bls_directory_content = download_url(
    BLS_URL
)

html = bls_directory_content.decode(
    "utf-8",
    errors="replace"
)

print(html[:1000])

<html><head><title>download.bls.gov - /pub/time.series/pr/</title></head><body><H1>download.bls.gov - /pub/time.series/pr/</H1><hr>

<pre><A HREF="/pub/time.series/">[To Parent Directory]</A><br><br> 11/5/2026  8:30 AM          102 <A HREF="/pub/time.series/pr/pr.class">pr.class</A><br> 9/13/2022  3:52 PM          562 <A HREF="/pub/time.series/pr/pr.contacts">pr.contacts</A><br> 11/5/2026  8:30 AM      1943714 <A HREF="/pub/time.series/pr/pr.data.0.Current">pr.data.0.Current</A><br> 11/5/2026  8:30 AM      3882812 <A HREF="/pub/time.series/pr/pr.data.1.AllData">pr.data.1.AllData</A><br> 11/5/2026  8:30 AM          191 <A HREF="/pub/time.series/pr/pr.duration">pr.duration</A><br> 11/5/2026  8:30 AM           40 <A HREF="/pub/time.series/pr/pr.footnote">pr.footnote</A><br> 11/5/2026  8:30 AM          745 <A HREF="/pub/time.series/pr/pr.measure">pr.measure</A><br>  1/7/1994  2:53 PM          146 <A HREF="/pub/time.series/pr/pr.period">pr.period</A><br>11/18/2011  3:05 PM           79 <A


In [0]:
%python
class BLSLinkParser(HTMLParser):

    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(
        self,
        tag,
        attrs
    ):

        if tag.lower() != "a":
            return

        attributes = dict(attrs)

        href = attributes.get("href")

        if href:
            self.links.append(href)

In [0]:
%python
def discover_bls_files():

    directory_content = download_url(
        BLS_URL
    )

    directory_html = (
        directory_content.decode(
            "utf-8",
            errors="replace"
        )
    )

    parser = BLSLinkParser()

    parser.feed(
        directory_html
    )

    discovered_files = []

    base_path = urlparse(
        BLS_URL
    ).path

    for href in parser.links:

        absolute_url = urljoin(
            BLS_URL,
            href
        )

        parsed = urlparse(
            absolute_url
        )

        # Only allow BLS host
        if parsed.netloc != "download.bls.gov":
            continue

        # Only current BLS PR directory
        if not parsed.path.startswith(
            base_path
        ):
            continue

        relative_path = (
            parsed.path[
                len(base_path):
            ]
        )

        # Ignore empty path
        if not relative_path:
            continue

        # Ignore directories
        if relative_path.endswith("/"):
            continue

        # Ignore nested folders
        if "/" in relative_path:
            continue

        filename = Path(
            parsed.path
        ).name

        discovered_files.append({
            "filename": filename,
            "url": absolute_url
        })

    return discovered_files

In [0]:
%python
bls_files = discover_bls_files()

print(
    "Number of BLS files discovered:",
    len(bls_files)
)

for item in bls_files:
    print(
        item["filename"],
        "->",
        item["url"]
    )

Number of BLS files discovered: 12
pr.class -> https://download.bls.gov/pub/time.series/pr/pr.class
pr.contacts -> https://download.bls.gov/pub/time.series/pr/pr.contacts
pr.data.0.Current -> https://download.bls.gov/pub/time.series/pr/pr.data.0.Current
pr.data.1.AllData -> https://download.bls.gov/pub/time.series/pr/pr.data.1.AllData
pr.duration -> https://download.bls.gov/pub/time.series/pr/pr.duration
pr.footnote -> https://download.bls.gov/pub/time.series/pr/pr.footnote
pr.measure -> https://download.bls.gov/pub/time.series/pr/pr.measure
pr.period -> https://download.bls.gov/pub/time.series/pr/pr.period
pr.seasonal -> https://download.bls.gov/pub/time.series/pr/pr.seasonal
pr.sector -> https://download.bls.gov/pub/time.series/pr/pr.sector
pr.series -> https://download.bls.gov/pub/time.series/pr/pr.series
pr.txt -> https://download.bls.gov/pub/time.series/pr/pr.txt


In [0]:
%python
def sha256_bytes(content):

    return hashlib.sha256(
        content
    ).hexdigest()

In [0]:
%python
def atomic_write(
    path,
    content
):

    temporary_path = path.with_name(
        f".{path.name}."
        f"{uuid.uuid4().hex}.tmp"
    )

    with open(
        temporary_path,
        "wb"
    ) as output_file:

        output_file.write(
            content
        )

        output_file.flush()

        os.fsync(
            output_file.fileno()
        )

    os.replace(
        temporary_path,
        path
    )

In [0]:
%python
download_results = []


for item in bls_files:

    filename = item["filename"]
    url = item["url"]

    print(
        f"Processing {filename}..."
    )

    content = download_url(
        url
    )

    new_hash = sha256_bytes(
        content
    )

    target_path = (
        BLS_RAW_DIR / filename
    )


    # -------------------------------
    # Check existing file
    # -------------------------------

    if target_path.exists():

        existing_content = (
            target_path.read_bytes()
        )

        existing_hash = (
            sha256_bytes(
                existing_content
            )
        )

    else:

        existing_hash = None


    # -------------------------------
    # Compare
    # -------------------------------

    if existing_hash == new_hash:

        status = "unchanged"

    else:

        atomic_write(
            target_path,
            content
        )

        if existing_hash is None:
            status = "new"
        else:
            status = "changed"


    # -------------------------------
    # Record result
    # -------------------------------

    download_results.append({

        "filename": filename,

        "status": status,

        "size_bytes": len(content),

        "sha256": new_hash,

        "volume_path": str(
            target_path
        )

    })

    print(
        f"   {status}"
    )

Processing pr.class...
   new
Processing pr.contacts...
   new
Processing pr.data.0.Current...
   new
Processing pr.data.1.AllData...
   new
Processing pr.duration...
   new
Processing pr.footnote...
   new
Processing pr.measure...
   new
Processing pr.period...
   new
Processing pr.seasonal...
   new
Processing pr.sector...
   new
Processing pr.series...
   new
Processing pr.txt...
   new


In [0]:
%python
bls_result_df = (
    spark.createDataFrame(
        download_results
    )
)

display(bls_result_df)

filename,sha256,size_bytes,status,volume_path
pr.class,ed5e7641b5b990460f02a3189b22b48535abcdf3eb505ebfe820ef4c7a288bc9,102,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.class
pr.contacts,6dfd5df62764766e9afe10b2de7ae8e1b8d372e44c2809f5bf292ee788b2d30a,562,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.contacts
pr.data.0.Current,96b35c3f6524daf118a9eb80e3bc24d0bd37357b78cfa3c16761fc85f666cfae,1943714,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.data.0.Current
pr.data.1.AllData,8b58755f43a8c63c5454e48ca10e9f2cae25d1f7acf6a35d89bddf0c7aeae08b,3882812,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.data.1.AllData
pr.duration,1f1b82965d6c9055b5e39952164b0c9fdc56630fe7cb8dfc0e19b3a7fcce8b10,191,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.duration
pr.footnote,29f9274a6e9da7dfc5e2258144d9edbaed34cbe02d852c82aab20c2e8ac3148b,40,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.footnote
pr.measure,459c07d625b60430028569cf37f20c4b618f3de0ece5756ef6fa034cbeb594c6,745,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.measure
pr.period,2a73359647de19a1fc85b2db5002d7efdda291005a501bfe432749c60bce0b2c,146,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.period
pr.seasonal,5ebe468ab3cbcfdd6b19b81f34af9281dd4728e891d78192f35de3a5bd9d7b24,79,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.seasonal
pr.sector,1eaef7fb47ac5accc468ff39c4e8916f575a644a009ee66ef8dbd188fe54e369,263,new,/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.sector


In [0]:
%python
display(
    dbutils.fs.ls(
        str(BLS_RAW_DIR)
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.class,pr.class,102,1794727351000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.contacts,pr.contacts,562,1794727351000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.data.0.Current,pr.data.0.Current,1943714,1794727352000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.data.1.AllData,pr.data.1.AllData,3882812,1794727352000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.duration,pr.duration,191,1794727353000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.footnote,pr.footnote,40,1794727353000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.measure,pr.measure,745,1794727353000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.period,pr.period,146,1794727354000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.seasonal,pr.seasonal,79,1794727354000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.sector,pr.sector,263,1794727354000


In [0]:
%python
population_content = download_url(
    POPULATION_URL
)

population_payload = json.loads(
    population_content.decode(
        "utf-8"
    )
)

print(
    "Top-level JSON keys:",
    population_payload.keys()
)

print(
    "Population records:",
    len(
        population_payload.get(
            "data",
            []
        )
    )
)

Top-level JSON keys: dict_keys(['annotations', 'page', 'columns', 'data'])
Population records: 11


In [0]:
%python
for row in population_payload[
    "data"
][:5]:

    print(row)

{'Nation ID': '01000US', 'Nation': 'United States', 'Year': 2013, 'Population': 316128839.0}
{'Nation ID': '01000US', 'Nation': 'United States', 'Year': 2014, 'Population': 318857056.0}
{'Nation ID': '01000US', 'Nation': 'United States', 'Year': 2015, 'Population': 321418821.0}
{'Nation ID': '01000US', 'Nation': 'United States', 'Year': 2016, 'Population': 323127515.0}
{'Nation ID': '01000US', 'Nation': 'United States', 'Year': 2017, 'Population': 325719178.0}


In [0]:
%python
population_path = (
    POPULATION_RAW_DIR
    / "population.json"
)

population_hash = sha256_bytes(
    population_content
)


if population_path.exists():

    existing_hash = sha256_bytes(
        population_path.read_bytes()
    )

else:

    existing_hash = None


if existing_hash == population_hash:

    population_status = "unchanged"

else:

    atomic_write(
        population_path,
        population_content
    )

    if existing_hash is None:
        population_status = "new"
    else:
        population_status = "changed"


print(
    "Population status:",
    population_status
)

print(
    "Population location:",
    population_path
)

Population status: new
Population location: /Volumes/workspace/bls_productivity/raw_files/current/population/population.json


In [0]:
%python
population_path = (
    POPULATION_RAW_DIR
    / "population.json"
)

population_hash = sha256_bytes(
    population_content
)


if population_path.exists():

    existing_hash = sha256_bytes(
        population_path.read_bytes()
    )

else:

    existing_hash = None


if existing_hash == population_hash:

    population_status = "unchanged"

else:

    atomic_write(
        population_path,
        population_content
    )

    if existing_hash is None:
        population_status = "new"
    else:
        population_status = "changed"


print(
    "Population status:",
    population_status
)

print(
    "Population location:",
    population_path
)

Population status: unchanged
Population location: /Volumes/workspace/bls_productivity/raw_files/current/population/population.json


In [0]:
%python
display(
    dbutils.fs.ls(
        "/Volumes/workspace/"
        "bls_productivity/"
        "raw_files/current/bls"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.class,pr.class,102,1794727351000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.contacts,pr.contacts,562,1794727351000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.data.0.Current,pr.data.0.Current,1943714,1794727352000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.data.1.AllData,pr.data.1.AllData,3882812,1794727352000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.duration,pr.duration,191,1794727353000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.footnote,pr.footnote,40,1794727353000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.measure,pr.measure,745,1794727353000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.period,pr.period,146,1794727354000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.seasonal,pr.seasonal,79,1794727354000
dbfs:/Volumes/workspace/bls_productivity/raw_files/current/bls/pr.sector,pr.sector,263,1794727354000
